# A/B Testing Practice: Interpretation and Decision-Making

This notebook is the practice companion to `01_AB_testing.ipynb` and `02_AB_testing_experiment_design.ipynb`.
It is designed to help you rehearse the workflow from those notebooks with a stronger focus on **interpretation**, **decision-making**, and **communicating results clearly**.

How to use this notebook:

- Try each exercise cell first.
- Keep the `TODO` markers until you are ready to solve them.
- Check the `### Solution` section right after each exercise.
- Compare not only the code, but also the interpretation.


In [1]:
import math

import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize


def clean_experiment_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Apply the same cleaning logic used in the first two notebooks."""
    users_both_groups = raw_df.groupby("user_id")["group"].nunique()
    users_both_groups = users_both_groups[users_both_groups > 1].index

    users_both_pages = raw_df.groupby("user_id")["landing_page"].nunique()
    users_both_pages = users_both_pages[users_both_pages > 1].index

    contaminated_users = set(users_both_groups) | set(users_both_pages)

    cleaned_df = raw_df[~raw_df["user_id"].isin(contaminated_users)].copy()
    first_exposure = cleaned_df.groupby("user_id")["timestamp"].min().reset_index()
    cleaned_df = cleaned_df.merge(first_exposure, on=["user_id", "timestamp"])
    cleaned_df["timestamp"] = pd.to_datetime(cleaned_df["timestamp"])
    return cleaned_df


def required_sample_size_per_variant(
    baseline_rate: float,
    relative_lift: float,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """Return the required users per variant for a two-sample conversion test."""
    power_model = NormalIndPower()
    target_rate = baseline_rate * (1 + relative_lift)
    effect_size = proportion_effectsize(target_rate, baseline_rate)
    required_n = power_model.solve_power(
        effect_size=effect_size,
        alpha=alpha,
        power=power,
        ratio=1.0,
        alternative="two-sided",
    )
    return math.ceil(required_n)


def runtime_days(
    sample_size_per_variant: int,
    avg_daily_total: float,
    traffic_split: float = 0.5,
) -> float:
    """Convert a sample-size target into expected test duration."""
    return sample_size_per_variant / (avg_daily_total * traffic_split)


raw_df = pd.read_csv("data/ab_data.csv")
official_clean_df = clean_experiment_data(raw_df)
daily_traffic = official_clean_df.groupby(official_clean_df["timestamp"].dt.date).size()
control_df = official_clean_df[official_clean_df["group"] == "control"].copy()
treatment_df = official_clean_df[official_clean_df["group"] == "treatment"].copy()
baseline_rate = control_df["converted"].mean()
avg_daily_total = daily_traffic.mean()

print(f"Raw rows loaded: {raw_df.shape[0]:,}")
print(f"Clean rows available for solutions: {official_clean_df.shape[0]:,}")

Raw rows loaded: 294,478
Clean rows available for solutions: 286,691


## Exercise 1: Read the Dataset Like an Analyst

**Learning goal:** Understand the raw data before doing any statistical work.

**Task:**
1. Identify the number of rows in the raw dataset.
2. Fill in the number of unique users.
3. Write a short description for each column.

**Expected output:** A few summary values plus a small table or dictionary describing the columns.


In [2]:
# Exercise 1 (my answer): count each user_id only once
unique_users = raw_df["user_id"].nunique()

# A plain-words description of every column
column_descriptions = {
    "user_id": "ID number of one visitor (can repeat in the raw log)",
    "timestamp": "Date and time the visitor first saw a page in the test",
    "group": "Bucket the visitor was put in: control (A) or treatment (B)",
    "landing_page": "Page the visitor actually saw: old_page or new_page",
    "converted": "1 if the visitor bought something, 0 if not",
}

print(f"Raw rows: {raw_df.shape[0]:,}")
print(f"Unique users: {unique_users:,}")
print(f"Rows that repeat a user: {raw_df.shape[0] - unique_users:,}")
display(
    pd.DataFrame(
        column_descriptions.items(), columns=pd.Index(["column", "description"])
    )
)
raw_df.head()


Raw rows: 294,478
Unique users: 290,584
Rows that repeat a user: 3,894


,column,description
0,user_id,ID number of one visitor (can repeat in the ra...
1,timestamp,Date and time the visitor first saw a page in ...
2,group,Bucket the visitor was put in: control (A) or ...
3,landing_page,Page the visitor actually saw: old_page or new...
4,converted,"1 if the visitor bought something, 0 if not"


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


### Solution


In [3]:
solution_column_descriptions = {
    "user_id": "Unique identifier for a user in the experiment",
    "timestamp": "Time of the user's recorded exposure",
    "group": "Assigned experiment group: control or treatment",
    "landing_page": "Page version the user saw: old_page or new_page",
    "converted": "Binary outcome showing whether the user converted",
}

print(f"Raw rows: {raw_df.shape[0]:,}")
print(f"Unique users: {raw_df['user_id'].nunique():,}")
display(
    pd.DataFrame(
        solution_column_descriptions.items(),
        columns=pd.Index(["column", "description"]),
    )
)
raw_df.head()

Raw rows: 294,478
Unique users: 290,584


,column,description
0,user_id,Unique identifier for a user in the experiment
1,timestamp,Time of the user's recorded exposure
2,group,Assigned experiment group: control or treatment
3,landing_page,Page version the user saw: old_page or new_page
4,converted,Binary outcome showing whether the user converted


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


## Exercise 2: Rebuild the Cleaning Logic

**Learning goal:** Recreate the core data-cleaning step used in the earlier notebooks.

**Task:**
1. Find users who appear in more than one group.
2. Find users who see more than one landing page.
3. Remove those contaminated users.
4. Keep only each user's first exposure.

**Expected output:** A cleaned dataframe and a short summary of cleaned rows and unique users.


In [4]:
# Exercise 2 (my answer)
# 1. Users whose rows show more than one group
groups_per_user = raw_df.groupby("user_id")["group"].nunique()
users_both_groups = groups_per_user[groups_per_user > 1].index

# 2. Users whose rows show more than one landing page
pages_per_user = raw_df.groupby("user_id")["landing_page"].nunique()
users_both_pages = pages_per_user[pages_per_user > 1].index

# 3. Either problem makes a user "contaminated": drop all their rows
contaminated_users = set(users_both_groups) | set(users_both_pages)
cleaned_practice_df = raw_df[~raw_df["user_id"].isin(contaminated_users)]

# 4. Sort by time and keep only each user's first row
cleaned_practice_df = (
    cleaned_practice_df.sort_values("timestamp")
    .drop_duplicates(subset="user_id", keep="first")
    .reset_index(drop=True)
)

print(f"Users in more than one group: {len(users_both_groups):,}")
print(f"Users who saw more than one page: {len(users_both_pages):,}")
print(f"Contaminated users removed: {len(contaminated_users):,}")
print(f"Rows after cleaning: {cleaned_practice_df.shape[0]:,}")
print(f"Unique users after cleaning: {cleaned_practice_df['user_id'].nunique():,}")
same = cleaned_practice_df.shape[0] == official_clean_df.shape[0]
print(f"Same row count as the notebooks' cleaning: {same}")


Users in more than one group: 1,895
Users who saw more than one page: 1,998
Contaminated users removed: 3,893
Rows after cleaning: 286,691
Unique users after cleaning: 286,691
Same row count as the notebooks' cleaning: True


### Solution


In [5]:
users_both_groups = raw_df.groupby("user_id")["group"].nunique()
users_both_groups = users_both_groups[users_both_groups > 1].index

users_both_pages = raw_df.groupby("user_id")["landing_page"].nunique()
users_both_pages = users_both_pages[users_both_pages > 1].index

contaminated_users = set(users_both_groups) | set(users_both_pages)

cleaned_practice_df = raw_df[~raw_df["user_id"].isin(contaminated_users)].copy()
first_exposure = cleaned_practice_df.groupby("user_id")["timestamp"].min().reset_index()
cleaned_practice_df = cleaned_practice_df.merge(
    first_exposure, on=["user_id", "timestamp"]
)
cleaned_practice_df["timestamp"] = pd.to_datetime(cleaned_practice_df["timestamp"])

print(f"Users in more than one group: {len(users_both_groups):,}")
print(f"Users who saw more than one page: {len(users_both_pages):,}")
print(f"Rows after cleaning: {cleaned_practice_df.shape[0]:,}")
print(f"Unique users after cleaning: {cleaned_practice_df['user_id'].nunique():,}")
print(
    f"Matches the established cleaned row count: {cleaned_practice_df.shape[0] == official_clean_df.shape[0]}"
)

Users in more than one group: 1,895
Users who saw more than one page: 1,998
Rows after cleaning: 286,691
Unique users after cleaning: 286,691
Matches the established cleaned row count: True


## Exercise 3: Compare Control and Treatment Rates

**Learning goal:** Move from cleaned data to a practical business comparison.

**Task:**
1. Compute the conversion rate for the control group.
2. Compute the conversion rate for the treatment group.
3. Compute the relative difference between treatment and control.
4. Write one sentence saying whether the observed gap looks practically meaningful.

**Expected output:** A few numeric values and a short interpretation.


In [6]:
control_subset = official_clean_df[official_clean_df["group"] == "control"]
treatment_subset = official_clean_df[official_clean_df["group"] == "treatment"]

# Exercise 3 (my answer)
# converted is 0/1, so the mean is the share of buyers
control_rate = control_subset["converted"].mean()
treatment_rate = treatment_subset["converted"].mean()

# Relative gap: change compared with control, as a share of control
relative_gap = (treatment_rate - control_rate) / control_rate
# Absolute gap in percentage points, for scale
absolute_gap_pp = (treatment_rate - control_rate) * 100

interpretation = (
    f"The new page converts {abs(absolute_gap_pp):.2f} percentage points "
    "LOWER than the old page (a relative drop of "
    f"{abs(relative_gap):.1%}). That is tiny and points the wrong way, "
    "so it is not a practically meaningful improvement."
)

print(f"Control conversion rate: {control_rate:.2%}")
print(f"Treatment conversion rate: {treatment_rate:.2%}")
print(f"Relative gap: {relative_gap:.2%}")
print(f"Absolute gap: {absolute_gap_pp:.2f} percentage points")
print(interpretation)


Control conversion rate: 12.02%
Treatment conversion rate: 11.87%
Relative gap: -1.20%
Absolute gap: -0.14 percentage points
The new page converts 0.14 percentage points LOWER than the old page (a relative drop of 1.2%). That is tiny and points the wrong way, so it is not a practically meaningful improvement.


### Solution


In [7]:
control_rate = control_subset["converted"].mean()
treatment_rate = treatment_subset["converted"].mean()
relative_gap = (treatment_rate - control_rate) / control_rate

print(f"Control conversion rate: {control_rate:.2%}")
print(f"Treatment conversion rate: {treatment_rate:.2%}")
print(f"Relative gap: {relative_gap:.2%}")

interpretation = (
    "The treatment page performs slightly worse than control, and the gap is small "
    "in absolute terms, so it does not look like a strong practical improvement."
)
print(interpretation)

Control conversion rate: 12.02%
Treatment conversion rate: 11.87%
Relative gap: -1.20%
The treatment page performs slightly worse than control, and the gap is small in absolute terms, so it does not look like a strong practical improvement.


## Exercise 4: Reconstruct a Frequentist Test

**Learning goal:** Connect conversion rates to a formal hypothesis test.

**Task:**
1. Build a 2x2 contingency table for control vs treatment and converted vs not converted.
2. Run a chi-square test.
3. Explain what the p-value means.
4. Explain one thing the p-value does **not** mean.

**Expected output:** A contingency table, the chi-square statistic, the p-value, and a short interpretation.


In [8]:
# Exercise 4 (my answer)
# 2x2 table: rows = group, columns = bought / did not buy
contingency_table = pd.crosstab(
    official_clean_df["group"], official_clean_df["converted"]
)
contingency_table.columns = ["not_converted", "converted"]

# Chi-square test without the small-sample (Yates) correction
chi2_stat, p_value, dof, expected = chi2_contingency(
    contingency_table, correction=False
)

print("Contingency table:")
print(contingency_table)
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(
    f"What it means: if both pages truly converted equally, we would see a "
    f"gap at least this big in about {p_value:.0%} of repeated experiments. "
    "That is common, so we cannot reject H0 at alpha = 0.05."
)
print(
    "What it does NOT mean: it is not the probability that H0 is true, "
    "and a large p-value does not prove the two pages are identical."
)


Contingency table:
           not_converted  converted
group                              
control           126073      17220
treatment         126373      17025
Chi-square statistic: 1.4284
p-value: 0.2320
What it means: if both pages truly converted equally, we would see a gap at least this big in about 23% of repeated experiments. That is common, so we cannot reject H0 at alpha = 0.05.
What it does NOT mean: it is not the probability that H0 is true, and a large p-value does not prove the two pages are identical.


### Solution


In [9]:
control_converted = int(control_df["converted"].sum())
control_not_converted = int(control_df["converted"].count() - control_converted)
treatment_converted = int(treatment_df["converted"].sum())
treatment_not_converted = int(treatment_df["converted"].count() - treatment_converted)

contingency_table = pd.DataFrame(
    {
        "converted": [control_converted, treatment_converted],
        "not_converted": [control_not_converted, treatment_not_converted],
    },
    index=pd.Index(["control", "treatment"]),
)

chi2_stat, p_value, _, _ = chi2_contingency(contingency_table, correction=False)

display(contingency_table)
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {p_value:.4f}")

display(
    Markdown(
        "The p-value measures how surprising a difference this large would be if "
        "there were truly no conversion difference between the groups. It does not "
        "measure the probability that the null hypothesis is true, and it does not "
        "prove that the two versions are identical."
    )
)

,converted,not_converted
control,17220,126073
treatment,17025,126373


Chi-square statistic: 1.4284
p-value: 0.2320


The p-value measures how surprising a difference this large would be if there were truly no conversion difference between the groups. It does not measure the probability that the null hypothesis is true, and it does not prove that the two versions are identical.

## Exercise 5: Plan a Future Experiment

**Learning goal:** Turn a business target into a sample-size and runtime requirement.

**Task:**
1. Assume the business wants to detect a **2% relative lift**.
2. Use the helper functions to estimate the required users per variant.
3. Estimate the runtime using the observed average daily traffic.
4. Decide whether that target looks feasible with the traffic in this dataset.

**Expected output:** A target-lift summary and a short feasibility statement.


In [10]:
target_relative_lift = 0.02

# Exercise 5 (my answer): helper functions from the setup cell
required_users = required_sample_size_per_variant(
    baseline_rate, target_relative_lift
)
estimated_days = runtime_days(required_users, avg_daily_total)

# What the dataset actually has (smaller group, to be safe)
observed_users = min(control_df.shape[0], treatment_df.shape[0])
observed_days = daily_traffic.shape[0]

if observed_users >= required_users:
    feasibility_statement = "Feasible: the dataset already has enough users."
else:
    feasibility_statement = (
        f"Not feasible in the observed window: we need {required_users:,} "
        f"users per variant but have {observed_users:,} "
        f"({observed_users / required_users:.0%} of the need). "
        f"At today's traffic the test would run about {estimated_days:.0f} "
        f"days instead of the {observed_days} days observed."
    )

print(f"Target relative lift: {target_relative_lift:.0%}")
print(f"Required users per variant: {required_users:,}")
print(f"Estimated runtime in days: {estimated_days:.1f}")
print(feasibility_statement)


Target relative lift: 2%
Required users per variant: 289,791
Estimated runtime in days: 46.5
Not feasible in the observed window: we need 289,791 users per variant but have 143,293 (49% of the need). At today's traffic the test would run about 46 days instead of the 23 days observed.


### Solution


In [11]:
target_relative_lift = 0.02
required_users = required_sample_size_per_variant(baseline_rate, target_relative_lift)
estimated_days = runtime_days(required_users, avg_daily_total)
observed_users_per_variant = min(control_df.shape[0], treatment_df.shape[0])

print(f"Target relative lift: {target_relative_lift:.0%}")
print(f"Required users per variant: {required_users:,}")
print(f"Estimated runtime in days: {estimated_days:.1f}")
print(f"Observed users per variant in this dataset: {observed_users_per_variant:,}")

if observed_users_per_variant >= required_users:
    feasibility_statement = "This looks feasible within the observed experiment size."
else:
    feasibility_statement = (
        "This target is not fully feasible within the observed experiment size; "
        "the dataset does not contain enough users per variant to detect a 2% lift "
        "with the chosen alpha and power."
    )

print(feasibility_statement)

Target relative lift: 2%
Required users per variant: 289,791
Estimated runtime in days: 46.5
Observed users per variant in this dataset: 143,293
This target is not fully feasible within the observed experiment size; the dataset does not contain enough users per variant to detect a 2% lift with the chosen alpha and power.


## Exercise 6: Compare Two Business Scenarios

**Learning goal:** Translate statistical constraints into a product recommendation.

**Task:**
Compare these two cases:

- Scenario A: the business only cares about a **1% relative lift**
- Scenario B: the business cares about a **3% relative lift**

For each scenario, estimate users per variant and runtime. Then write a short recommendation about which scenario is more realistic for the current traffic level.

**Expected output:** A two-row comparison table plus a recommendation.


In [12]:
scenario_lifts = [0.01, 0.03]
scenario_rows = []

# Exercise 6 (my answer): one row per business target
for lift in scenario_lifts:
    n_needed = required_sample_size_per_variant(baseline_rate, lift)
    scenario_rows.append(
        {
            "relative_lift": lift,
            "required_users_per_variant": n_needed,
            "estimated_runtime_days": round(
                runtime_days(n_needed, avg_daily_total), 1
            ),
        }
    )

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

one_pct, three_pct = scenario_df["estimated_runtime_days"]
print(
    f"Recommendation: a 1% lift needs about {one_pct:.0f} days, which is "
    f"over half a year of traffic. A 3% lift needs about {three_pct:.0f} "
    "days, roughly a month. Plan for the 3% target, or only test "
    "changes bold enough to move conversion by 3% or more."
)


,relative_lift,required_users_per_variant,estimated_runtime_days
0,0.01,1154231,185.2
1,0.03,129342,20.8


Recommendation: a 1% lift needs about 185 days, which is over half a year of traffic. A 3% lift needs about 21 days, roughly a month. Plan for the 3% target, or only test changes bold enough to move conversion by 3% or more.


### Solution


In [13]:
scenario_lifts = [0.01, 0.03]
scenario_rows = []

for lift in scenario_lifts:
    required_users = required_sample_size_per_variant(baseline_rate, lift)
    estimated_days = runtime_days(required_users, avg_daily_total)
    scenario_rows.append(
        {
            "relative_lift": f"{lift:.0%}",
            "required_users_per_variant": required_users,
            "estimated_runtime_days": round(estimated_days, 1),
        }
    )

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

display(
    Markdown(
        "A **1%** lift target is much harder to detect and would require a very long "
        "experiment. A **3%** lift target is more realistic for the current traffic "
        "because the required sample size and runtime are much smaller. If the business "
        "needs fast decisions, the 3% scenario is the more practical planning target."
    )
)

,relative_lift,required_users_per_variant,estimated_runtime_days
0,1%,1154231,185.2
1,3%,129342,20.8


A **1%** lift target is much harder to detect and would require a very long experiment. A **3%** lift target is more realistic for the current traffic because the required sample size and runtime are much smaller. If the business needs fast decisions, the 3% scenario is the more practical planning target.

## Exercise 7: Optional Challenge - Choose the Right Workflow

**Learning goal:** Decide when to use each notebook in a real project.

**Task:**
Imagine a product manager asks, *"Should we launch a new design experiment next month, and how would we evaluate it?"*

Write a short answer describing:

1. Which notebook you would start with first
2. Which notebook you would use after data collection
3. Why that order makes sense

**Expected output:** A short written recommendation, ideally 3-5 sentences.


In [14]:
# Exercise 7 (my answer)
starting_notebook = "02_AB_testing_experiment_design.ipynb"
follow_up_notebook = "01_AB_testing.ipynb"
reasoning = (
    "First plan: pick the smallest lift worth acting on, then compute users "
    "per variant and runtime, so we know before launch whether the test can "
    "give an answer. Then run the test for the full planned time without "
    "peeking. After data collection, clean the data, compare the groups and "
    "run the chi-square test (and the Bayesian view). Planning first stops "
    "us from running an underpowered test and misreading its result."
)

print(f"Start with: {starting_notebook}")
print(f"Then use: {follow_up_notebook}")
print(reasoning)


Start with: 02_AB_testing_experiment_design.ipynb
Then use: 01_AB_testing.ipynb
First plan: pick the smallest lift worth acting on, then compute users per variant and runtime, so we know before launch whether the test can give an answer. Then run the test for the full planned time without peeking. After data collection, clean the data, compare the groups and run the chi-square test (and the Bayesian view). Planning first stops us from running an underpowered test and misreading its result.


### Solution


In [15]:
starting_notebook = "02_AB_testing_experiment_design.ipynb"
follow_up_notebook = "01_AB_testing.ipynb"
reasoning = (
    "Start with the experiment-design notebook because it helps define whether the "
    "test is worth running, how much traffic is needed, and how long the experiment "
    "should last. After data collection, use the analysis notebook to clean the data, "
    "compare control and treatment, and interpret the statistical results. This order "
    "keeps the team from launching an underpowered test and then misreading the outcome."
)

print(f"Start with: {starting_notebook}")
print(f"Then use: {follow_up_notebook}")
print(reasoning)

Start with: 02_AB_testing_experiment_design.ipynb
Then use: 01_AB_testing.ipynb
Start with the experiment-design notebook because it helps define whether the test is worth running, how much traffic is needed, and how long the experiment should last. After data collection, use the analysis notebook to clean the data, compare control and treatment, and interpret the statistical results. This order keeps the team from launching an underpowered test and then misreading the outcome.


## Wrap-Up

By the end of this notebook, you should be able to:

- inspect the raw experiment data
- reproduce the cleaning logic
- compare conversion performance
- interpret a frequentist test carefully
- plan a future experiment using sample size and runtime estimates
- choose the right workflow for a real product question
